In [1]:
!tar -xzvf ../data/hwu.tar.gz

x hwu/
x hwu/categories.json
x hwu/train_5.csv
x hwu/train_10.csv
x hwu/val.csv
x hwu/test.csv
x hwu/train.csv


In [2]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

# Chuẩn bị dữ liệu

In [3]:
# Dữ liệu có thể được phân tách bằng tab và không có header
df_train = pd.read_csv('hwu/train.csv', sep=',', header=0, names=['text', 'intent'])
df_val = pd.read_csv('hwu/val.csv', sep=',', header=0, names=['text', 'intent'])
df_test = pd.read_csv('hwu/test.csv', sep=',', header=0, names=['text', 'intent'])
print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)
df_train.head()

Train shape: (8954, 2)
Validation shape: (1076, 2)
Test shape: (1076, 2)


,text,intent
0,what alarms do i have set right now,alarm_query
1,checkout today alarm of meeting,alarm_query
2,report alarm settings,alarm_query
3,see see for me the alarms that you have set to...,alarm_query
4,is there an alarm for ten am,alarm_query


In [4]:
label_encoder = LabelEncoder()
df_train['intent'] = label_encoder.fit_transform(df_train['intent'])
df_val['intent'] = label_encoder.transform(df_val['intent'])
df_test['intent'] = label_encoder.transform(df_test['intent'])

num_classes = len(label_encoder.classes_)
print(f"Number of classes: {num_classes}")
label_encoder.classes_

Number of classes: 64


array(['alarm_query', 'alarm_remove', 'alarm_set', 'audio_volume_down',
       'audio_volume_mute', 'audio_volume_up', 'calendar_query',
       'calendar_remove', 'calendar_set', 'cooking_recipe',
       'datetime_convert', 'datetime_query', 'email_addcontact',
       'email_query', 'email_querycontact', 'email_sendemail',
       'general_affirm', 'general_commandstop', 'general_confirm',
       'general_dontcare', 'general_explain', 'general_joke',
       'general_negate', 'general_praise', 'general_quirky',
       'general_repeat', 'iot_cleaning', 'iot_coffee',
       'iot_hue_lightchange', 'iot_hue_lightdim', 'iot_hue_lightoff',
       'iot_hue_lighton', 'iot_hue_lightup', 'iot_wemo_off',
       'iot_wemo_on', 'lists_createoradd', 'lists_query', 'lists_remove',
       'music_likeness', 'music_query', 'music_settings', 'news_query',
       'play_audiobook', 'play_game', 'play_music', 'play_podcasts',
       'play_radio', 'qa_currency', 'qa_definition', 'qa_factoid',
       'qa_maths'

In [5]:
df_train.head()

,text,intent
0,what alarms do i have set right now,0
1,checkout today alarm of meeting,0
2,report alarm settings,0
3,see see for me the alarms that you have set to...,0
4,is there an alarm for ten am,0


# Task 1: Pipeline TF-IDF + Logistic Regression

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

## Tạo pipeline

In [7]:
tfidf_lr_pipeline = make_pipeline(
    TfidfVectorizer(max_features=5000),
    LogisticRegression(max_iter=1000)
)

## Huấn luyện và đánh giá mô hình

In [8]:
tfidf_lr_pipeline.fit(df_train['text'], df_train['intent'])
y_pred_baseline1 = tfidf_lr_pipeline.predict(df_test['text'])
print(classification_report(df_test['intent'], y_pred_baseline1, target_names=label_encoder.classes_))

                          precision    recall  f1-score   support

             alarm_query       0.90      0.95      0.92        19
            alarm_remove       1.00      0.73      0.84        11
               alarm_set       0.77      0.89      0.83        19
       audio_volume_down       1.00      0.75      0.86         8
       audio_volume_mute       0.92      0.80      0.86        15
         audio_volume_up       0.93      1.00      0.96        13
          calendar_query       0.45      0.53      0.49        19
         calendar_remove       0.89      0.89      0.89        19
            calendar_set       0.87      0.68      0.76        19
          cooking_recipe       0.59      0.68      0.63        19
        datetime_convert       0.67      0.75      0.71         8
          datetime_query       0.74      0.89      0.81        19
        email_addcontact       0.78      0.88      0.82         8
             email_query       0.83      0.79      0.81        19
      ema

# Task 2: Pipeline Word2Vec (Trung bình) + Dense Layer

In [9]:
import numpy as np
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [10]:
# 1. Huấn luyện mô hình Word2Vec với tham số tối ưu
sentences = [text.split() for text in df_train['text']]
w2v_model = Word2Vec(
    sentences, 
    vector_size=200,      
    window=5, 
    min_count=1, 
    workers=4,
    sg=1,                
    epochs=20             
)

In [11]:
# 2. Chuyển đổi câu thành vector trung bình
def sentence_to_avg_vector(text, model):
    """
    Chuyển đổi câu thành vector trung bình từ Word2Vec
    """
    words = text.split()
    word_vectors = []
    
    for word in words:
        if word in model.wv:
            word_vectors.append(model.wv[word])
    
    if len(word_vectors) == 0:
        # Nếu không có từ nào trong vocabulary, trả về vector 0
        return np.zeros(model.vector_size)
    
    # Tính vector trung bình
    avg_vector = np.mean(word_vectors, axis=0)
    return avg_vector

In [12]:
# 3. Tạo dữ liệu train/val/test X_train_avg, X_val_avg, X_test_avg
X_train_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_train['text']])
X_val_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_val['text']])
X_test_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_test['text']])

print("X_train_avg shape:", X_train_avg.shape)
print("X_val_avg shape:", X_val_avg.shape)
print("X_test_avg shape:", X_test_avg.shape)

X_train_avg shape: (8954, 200)
X_val_avg shape: (1076, 200)
X_test_avg shape: (1076, 200)


In [13]:
# 4. Xây dựng mô hình Sequential của Keras

model = Sequential([
    Dense(256, activation='relu', input_shape=(w2v_model.vector_size,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

In [14]:
# 5. Compile, huấn luyện và đánh giá mô hình

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

history = model.fit(
    X_train_avg, 
    df_train['intent'],
    validation_data=(X_val_avg, df_val['intent']),
    epochs=20,
    batch_size=32,
    verbose=1
)

# Đánh giá mô hình
test_loss, test_accuracy = model.evaluate(X_test_avg, df_test['intent'], verbose=0)
print(f"\nTest Accuracy: {test_accuracy:.4f}")

# Dự đoán và in classification report
y_pred = model.predict(X_test_avg, verbose=0)
y_pred_baseline2 = np.argmax(y_pred, axis=1)
print("\nClassification Report (Task 2):")
print(classification_report(df_test['intent'], y_pred_baseline2, target_names=label_encoder.classes_))

Epoch 1/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.4623 - loss: 2.1440 - val_accuracy: 0.6264 - val_loss: 2.2165
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.4623 - loss: 2.1440 - val_accuracy: 0.6264 - val_loss: 2.2165
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6604 - loss: 1.2254 - val_accuracy: 0.7230 - val_loss: 0.9610
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6604 - loss: 1.2254 - val_accuracy: 0.7230 - val_loss: 0.9610
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6960 - loss: 1.0539 - val_accuracy: 0.7770 - val_loss: 0.7792
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6960 - loss: 1.0539 - val_accuracy: 0.7770 - val_loss: 0.7792
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7265 - loss: 0.9536 - val_accuracy: 0.7770 - val_loss: 0.7693
Epoch 5/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7265 - loss: 0.9536 - val_accuracy: 0.

# Task 3: Mô hình Nâng cao (Embedding Pre-trained + LSTM)

In [15]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM

In [16]:
# 1. Tiền xử lý cho mô hình chuỗi
# a. Tokenizer: Tạo vocab và chuyển text thành chuỗi chỉ số
vocab_size = 5000
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<UNK>")
tokenizer.fit_on_texts(df_train['text'])

train_sequences = tokenizer.texts_to_sequences(df_train['text'])
val_sequences = tokenizer.texts_to_sequences(df_val['text'])
test_sequences = tokenizer.texts_to_sequences(df_test['text'])

# b. Padding: Đảm bảo các chuỗi có cùng độ dài
max_len = 50
X_train_pad = pad_sequences(train_sequences, maxlen=max_len, padding='post')
X_val_pad = pad_sequences(val_sequences, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(test_sequences, maxlen=max_len, padding='post')

print("X_train_pad shape:", X_train_pad.shape)
print("X_val_pad shape:", X_val_pad.shape)
print("X_test_pad shape:", X_test_pad.shape)

X_train_pad shape: (8954, 50)
X_val_pad shape: (1076, 50)
X_test_pad shape: (1076, 50)


In [17]:
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = w2v_model.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

In [29]:
# 3. Xây dựng mô hình Sequential với LSTM
from tensorflow.keras.layers import Bidirectional, SpatialDropout1D

lstm_model_pretrained = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=False
    ),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(128, dropout=0.2, recurrent_dropout=0.2, return_sequences=True)),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

In [30]:
# 4. Compile, huấn luyện (sử dụng EarlyStopping) và đánh giá
from tensorflow.keras.optimizers import Adam

lstm_model_pretrained.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1,
    min_lr=1e-6
)

history_pretrained = lstm_model_pretrained.fit(
    X_train_pad,
    df_train['intent'],
    validation_data=(X_val_pad, df_val['intent']),
    epochs=20,
    batch_size=64,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# Đánh giá mô hình
test_loss, test_accuracy = lstm_model_pretrained.evaluate(X_test_pad, df_test['intent'], verbose=0)
print(f"\nTest Accuracy: {test_accuracy:.4f}")

# Dự đoán và in classification report
y_pred = lstm_model_pretrained.predict(X_test_pad, verbose=0)
y_pred_task3 = np.argmax(y_pred, axis=1)
print("\nClassification Report (Task 3 - Pre-trained Embedding + Bi-LSTM):")
print(classification_report(df_test['intent'], y_pred_task3, target_names=label_encoder.classes_))

Epoch 1/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 30s 168ms/step - accuracy: 0.3050 - loss: 2.7560 - val_accuracy: 0.5864 - val_loss: 3.2316 - learning_rate: 0.0010
Epoch 2/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 22s 158ms/step - accuracy: 0.5753 - loss: 1.5241 - val_accuracy: 0.7203 - val_loss: 1.7601 - learning_rate: 0.0010
Epoch 3/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 23s 163ms/step - accuracy: 0.6517 - loss: 1.2167 - val_accuracy: 0.7639 - val_loss: 0.9844 - learning_rate: 0.0010
Epoch 4/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 23s 166ms/step - accuracy: 0.6925 - loss: 1.0597 - val_accuracy: 0.7928 - val_loss: 0.7437 - learning_rate: 0.0010
Epoch 5/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 23s 164ms/step - accuracy: 0.7173 - loss: 0.9627 - val_accuracy: 0.7918 - val_loss: 0.6878 - learning_rate: 0.0010
Epoch 6/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 23s 167ms/step - accuracy: 0.7361 - loss: 0.8915 - val_accuracy: 0.7946 - val_loss: 0.6808 - learning_rate: 0.0010
Epoch 7/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 23s 165ms/step - accuracy: 0.7

## Task 4: Mô hình Nâng cao (Embedding học từ đầu + LSTM)

In [20]:
# Dữ liệu đã được tiền xử lý (tokenized, padded) từ nhiệm vụ 3
# 1. Xây dựng mô hình tối ưu với Bi-LSTM
lstm_model_scratch = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=200,  # Tăng chiều embedding
        input_length=max_len
    ),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(128, dropout=0.2, recurrent_dropout=0.2, return_sequences=True)),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

In [21]:
# 2. Compile, huấn luyện và đánh giá mô hình
lstm_model_scratch.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1,
    min_lr=1e-6
)

history_scratch = lstm_model_scratch.fit(
    X_train_pad,
    df_train['intent'],
    validation_data=(X_val_pad, df_val['intent']),
    epochs=10,
    batch_size=64,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# Đánh giá mô hình
test_loss, test_accuracy = lstm_model_scratch.evaluate(X_test_pad, df_test['intent'], verbose=0)
print(f"\nTest Accuracy: {test_accuracy:.4f}")

# Dự đoán và in classification report
y_pred = lstm_model_scratch.predict(X_test_pad, verbose=0)
y_pred_task4 = np.argmax(y_pred, axis=1)
print("\nClassification Report (Task 4 - Embedding from Scratch + Bi-LSTM):")
print(classification_report(df_test['intent'], y_pred_task4, target_names=label_encoder.classes_))

Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 34s 196ms/step - accuracy: 0.2535 - loss: 3.2129 - val_accuracy: 0.6069 - val_loss: 3.6222 - learning_rate: 0.0010
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 26s 183ms/step - accuracy: 0.7386 - loss: 1.0964 - val_accuracy: 0.8513 - val_loss: 2.0400 - learning_rate: 0.0010
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 25s 182ms/step - accuracy: 0.8688 - loss: 0.5337 - val_accuracy: 0.8745 - val_loss: 0.7635 - learning_rate: 0.0010
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 26s 182ms/step - accuracy: 0.9155 - loss: 0.3370 - val_accuracy: 0.8625 - val_loss: 0.5405 - learning_rate: 0.0010
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 25s 181ms/step - accuracy: 0.9438 - loss: 0.2276 - val_accuracy: 0.8736 - val_loss: 0.4791 - learning_rate: 0.0010
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 26s 182ms/step - accuracy: 0.9559 - loss: 0.1767 - val_accuracy: 0.8708 - val_loss: 0.5286 - learning_rate: 0.0010
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 27s 191ms/step - accuracy: 0.9

# Task 5: Đánh giá và so sánh

## Đánh giá định lượng

In [32]:
# Tính F1-score (macro) cho từng mô hình
from sklearn.metrics import f1_score

# Task 1: TF-IDF + Logistic Regression
f1_task1 = f1_score(df_test['intent'], y_pred_baseline1, average='macro')
loss_task1 = None  

# Task 2: Word2Vec (Avg) + Dense
f1_task2 = f1_score(df_test['intent'], y_pred_baseline2, average='macro')
loss_task2 = test_loss 

# Task 3: Embedding (Pre-trained) + LSTM
test_loss_task3, test_accuracy_task3 = lstm_model_pretrained.evaluate(X_test_pad, df_test['intent'], verbose=0)
f1_task3 = f1_score(df_test['intent'], y_pred_task3, average='macro')
loss_task3 = test_loss_task3

# Task 4: Embedding (Scratch) + LSTM
test_loss_task4, test_accuracy_task4 = lstm_model_scratch.evaluate(X_test_pad, df_test['intent'], verbose=0)
f1_task4 = f1_score(df_test['intent'], y_pred_task4, average='macro')
loss_task4 = test_loss_task4

# Tạo DataFrame để hiển thị bảng
results_df = pd.DataFrame({
    'Pipeline': [
        'TF-IDF + Logistic Regression',
        'Word2Vec (Avg) + Dense',
        'Embedding (Pre-trained) + LSTM',
        'Embedding (Scratch) + LSTM'
    ],
    'F1-score (Macro)': [f1_task1, f1_task2, f1_task3, f1_task4],
    'Test Loss': [loss_task1, loss_task2, loss_task3, loss_task4]
})

# Hiển thị bảng
print("\n" + "="*80)
print("BẢNG TỔNG HỢP KẾT QUẢ F1-SCORE (MACRO) VÀ LOSS TRÊN TẬP KIỂM TRA")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)



BẢNG TỔNG HỢP KẾT QUẢ F1-SCORE (MACRO) VÀ LOSS TRÊN TẬP KIỂM TRA
                      Pipeline  F1-score (Macro)  Test Loss
  TF-IDF + Logistic Regression          0.835298        NaN
        Word2Vec (Avg) + Dense          0.775115   0.616172
Embedding (Pre-trained) + LSTM          0.833110   0.616172
    Embedding (Scratch) + LSTM          0.842764   0.828690


## Đánh giá định tính

In [25]:
test = [
    "set an alarm for 7 am tomorrow",
    "turn the volume down a little",
    "mute the audio completely",
    "what meetings do I have next Wednesday",
    "add a doctor's appointment at 4 pm",
    "remove my lunch event on Friday",
    "how do I cook fried rice",
    "convert 2 pm PST to CET",
    "what day is December 12th",
    "show me my unread emails",
    "send an email to Michael saying I'm on my way",
    "tell me a funny joke",
    "dim the living room lights to 20 percent",
    "turn on the hue lights in the kitchen",
    "what’s on my grocery list",
    "play some relaxing jazz music",
    "define the word perspective",
    "how much is 100 dollars in euros",
    "what’s the latest news today",
    "will it rain in London tomorrow"
]


test_label = [
    "alarm_set",
    "audio_volume_down",
    "audio_volume_mute",
    "calendar_query",
    "calendar_set",
    "calendar_remove",
    "cooking_recipe",
    "datetime_convert",
    "datetime_query",
    "email_query",
    "email_sendemail",
    "general_joke",
    "iot_hue_lightdim",
    "iot_hue_lighton",
    "lists_query",
    "play_music",
    "qa_definition",
    "qa_currency",
    "news_query",
    "weather_query"
]


In [31]:
results_list = []

for idx, text in enumerate(test):
    true_label = test_label[idx]
    
    # Task 1: TF-IDF + Logistic Regression
    pred_1_label = label_encoder.classes_[tfidf_lr_pipeline.predict([text])[0]]
    
    # Task 2: Word2Vec (Avg) + Dense
    text_vec = sentence_to_avg_vector(text, w2v_model).reshape(1, -1)
    pred_2_label = label_encoder.classes_[np.argmax(model.predict(text_vec, verbose=0)[0])]
    
    # Task 3: Embedding (Pre-trained) + LSTM
    text_seq = tokenizer.texts_to_sequences([text])
    text_pad = pad_sequences(text_seq, maxlen=max_len, padding='post')
    pred_3_label = label_encoder.classes_[np.argmax(lstm_model_pretrained.predict(text_pad, verbose=0)[0])]
    
    # Task 4: Embedding (Scratch) + LSTM
    pred_4_label = label_encoder.classes_[np.argmax(lstm_model_scratch.predict(text_pad, verbose=0)[0])]
    
    results_list.append({
        'text': text,
        'true': true_label,
        'task1': pred_1_label,
        'task2': pred_2_label,
        'task3': pred_3_label,
        'task4': pred_4_label,
        'task1_correct': pred_1_label == true_label,
        'task2_correct': pred_2_label == true_label,
        'task3_correct': pred_3_label == true_label,
        'task4_correct': pred_4_label == true_label
    })

task1_correct = sum([r['task1_correct'] for r in results_list])
task2_correct = sum([r['task2_correct'] for r in results_list])
task3_correct = sum([r['task3_correct'] for r in results_list])
task4_correct = sum([r['task4_correct'] for r in results_list])
total = len(test)

print(f"\nTask 1 (TF-IDF + LR):              {task1_correct}/{total} đúng = {task1_correct/total*100:.1f}%")
print(f"Task 2 (Word2Vec + Dense):         {task2_correct}/{total} đúng = {task2_correct/total*100:.1f}%")
print(f"Task 3 (Pre-trained + Bi-LSTM):    {task3_correct}/{total} đúng = {task3_correct/total*100:.1f}%")
print(f"Task 4 (Scratch + Bi-LSTM):        {task4_correct}/{total} đúng = {task4_correct/total*100:.1f}%")



Task 1 (TF-IDF + LR):              19/20 đúng = 95.0%
Task 2 (Word2Vec + Dense):         18/20 đúng = 90.0%
Task 3 (Pre-trained + Bi-LSTM):    19/20 đúng = 95.0%
Task 4 (Scratch + Bi-LSTM):        20/20 đúng = 100.0%
